In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
%run ./UDF/udf_silver_incremental_ingest

In [0]:


src_bronze_path = "/Volumes/data_governance/bronze_cost_monitoring/job_run_timeline"
tgt_silver_table = "data_governance.silver_cost_monitoring.job_run"



In [0]:
df=silver_incremental_ingest(src_bronze_path,tgt_silver_table)

In [0]:
if df.count()==0:
    dbutils.notebook.exit("No new records to load")
else:
    pass

In [0]:
df.limit(20).display()

In [0]:

# Cast duration columns
df = df.withColumn("setup_duration_seconds", col("setup_duration_seconds").cast("long"))
df = df.withColumn("queue_duration_seconds", col("queue_duration_seconds").cast("long"))
df = df.withColumn("run_duration_seconds", col("run_duration_seconds").cast("long"))
df = df.withColumn("cleanup_duration_seconds", col("cleanup_duration_seconds").cast("long"))
df = df.withColumn("execution_duration_seconds", col("execution_duration_seconds").cast("long"))

# Clean categorical columns
df = df.withColumn("trigger_type", upper(trim(col("trigger_type"))))
df = df.withColumn("result_state", upper(trim(col("result_state"))))
df = df.withColumn("run_type", upper(trim(col("run_type"))))
df = df.withColumn("termination_type", upper(trim(col("termination_type"))))
df = df.withColumn("termination_code", upper(trim(col("termination_code"))))

# Derived runtime metric
df = df.withColumn(
    "job_total_runtime_seconds",
    col("run_duration_seconds")
)

df = df.withColumn("compute_item", explode_outer("compute"))
df = df.withColumn("compute_type", col("compute_item.type")) \
       .withColumn("cluster_id", col("compute_item.cluster_id")) \
       .withColumn("warehouse_id", col("compute_item.warehouse_id"))

df = df.drop("compute_item","compute")

df = df.dropDuplicates([
    "workspace_id",
    "job_id",
    "run_id"
])

In [0]:
df.display()

In [0]:

# Writing with Liquid Clustering
df.write\
 .format("delta")\
 .mode("append") \
 .partitionBy("result_state") \
 .saveAsTable(tgt_silver_table)

In [0]:
%sql
select * from data_governance.silver_cost_monitoring.cluster_node;